# Instruction-Based vs Span-Based Label Masking

This notebook visualizes how `DataCollatorForCompletionOnlyLM` masks labels for conversations that include **tool calls** and **tool results** under two modes:

1. **Instruction-based masking** (default) — uses `instruction_template` + `response_template` pairs via `zip()`.
2. **Span-based masking** (new) — uses `response_template` + `end_of_turn_template` to find assistant spans.

The key question: does instruction-based masking correctly handle tool-result tokens, or does it leak them into the training loss?

In [4]:
import html as html_mod
from IPython.display import HTML, display

from transformers import AutoTokenizer

from oumi.core.collators.trl_data_collator_for_completion_only_lm import (
    DataCollatorForCompletionOnlyLM,
)

IGNORE_INDEX = -100

## 1. Setup: Tokenizer and Chat Template\n\nWe use a ChatML-style model (Qwen2.5) so that tool calls use realistic special tokens.

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Template markers for ChatML
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
INSTRUCTION_TEMPLATE = "<|im_start|>user\n"
END_OF_TURN = "<|im_end|>"

print(f"Model: {MODEL_NAME}")
print(f"pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"Response template tokens: {tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)}")
print(f"Instruction template tokens: {tokenizer.encode(INSTRUCTION_TEMPLATE, add_special_tokens=False)}")
print(f"EOT tokens: {tokenizer.encode(END_OF_TURN, add_special_tokens=False)}")

Model: Qwen/Qwen2.5-0.5B-Instruct
pad_token: '<|endoftext|>' (id=151643)
eos_token: '<|im_end|>' (id=151645)
Response template tokens: [151644, 77091, 198]
Instruction template tokens: [151644, 872, 198]
EOT tokens: [151645]


## 2. Build Test Conversations\n\nWe create three scenarios:\n1. **Simple** — user/assistant only (no tools)\n2. **Tool call** — assistant calls a tool, tool returns a result, assistant gives final answer\n3. **Parallel tool calls** — assistant makes two tool calls before the final answer

In [6]:
# --- Scenario 1: Simple user/assistant (no tools) ---
simple_text = (
    "<|im_start|>user\n"
    "What is the capital of France?<|im_end|>\n"
    "<|im_start|>assistant\n"
    "The capital of France is Paris.<|im_end|>\n"
)

# --- Scenario 2: Single tool call ---
# Assistant calls get_weather, tool returns result, assistant gives final answer.
tool_call_text = (
    "<|im_start|>user\n"
    "What's the weather in Tokyo?<|im_end|>\n"
    "<|im_start|>assistant\n"
    '<tool_call>\n{"name": "get_weather", "arguments": {"city": "Tokyo"}}\n</tool_call><|im_end|>\n'
    "<|im_start|>tool\n"
    '{"temperature": 22, "condition": "sunny"}<|im_end|>\n'
    "<|im_start|>assistant\n"
    "The weather in Tokyo is 22°C and sunny.<|im_end|>\n"
)

# --- Scenario 3: Parallel tool calls ---
parallel_tool_text = (
    "<|im_start|>user\n"
    "Compare weather in Tokyo and Paris.<|im_end|>\n"
    "<|im_start|>assistant\n"
    '<tool_call>\n{"name": "get_weather", "arguments": {"city": "Tokyo"}}\n</tool_call>\n'
    '<tool_call>\n{"name": "get_weather", "arguments": {"city": "Paris"}}\n</tool_call><|im_end|>\n'
    "<|im_start|>tool\n"
    '{"temperature": 22, "condition": "sunny"}<|im_end|>\n'
    "<|im_start|>tool\n"
    '{"temperature": 18, "condition": "cloudy"}<|im_end|>\n'
    "<|im_start|>assistant\n"
    "Tokyo is 22°C and sunny. Paris is 18°C and cloudy.<|im_end|>\n"
)

scenarios = {
    "Simple (no tools)": simple_text,
    "Single tool call": tool_call_text,
    "Parallel tool calls": parallel_tool_text,
}

# Tokenize all scenarios
tokenized = {}
for name, text in scenarios.items():
    ids = tokenizer.encode(text, add_special_tokens=False)
    tokenized[name] = ids
    print(f"{name}: {len(ids)} tokens")

Simple (no tools): 24 tokens
Single tool call: 73 tokens
Parallel tool calls: 119 tokens


## 3. Visualization Helper\n\nColor-coded token display:\n- 🟩 **Green** = unmasked (label = token id, included in loss)\n- 🟥 **Red** = masked (label = -100, excluded from loss)\n\nTokens are shown with their decoded text. Hover for token ID.

In [7]:
def visualize_labels(token_ids: list[int], labels: list[int], title: str, tokenizer) -> str:
    """Build an HTML string showing each token colored by whether it's masked."""
    parts = [
        f"<h3>{html_mod.escape(title)}</h3>",
        '<div style="font-family: monospace; font-size: 13px; line-height: 2.2; '
        'background: #1e1e1e; color: #d4d4d4; padding: 12px; border-radius: 6px; '
        'white-space: pre-wrap; word-break: break-all;">',
    ]

    for tid, lbl in zip(token_ids, labels):
        decoded = tokenizer.decode([tid])
        escaped = html_mod.escape(decoded).replace("\n", "↵\n")
        if lbl == IGNORE_INDEX:
            # Masked — red background
            bg, fg = "#4a1a1a", "#ff8080"
        else:
            # Unmasked — green background
            bg, fg = "#1a4a1a", "#80ff80"
        parts.append(
            f'<span style="background:{bg}; color:{fg}; padding:1px 3px; '
            f'border-radius:3px; margin:1px;" '
            f'title="id={tid} label={lbl}">{escaped}</span>'
        )

    parts.append("</div>")
    return "".join(parts)


def show_comparison(name: str, token_ids: list[int], instr_labels: list[int], span_labels: list[int]):
    """Display side-by-side comparison of both masking modes."""
    h = '<div style="display:flex; gap:20px; flex-wrap:wrap;">'
    h += '<div style="flex:1; min-width:400px;">'
    h += visualize_labels(token_ids, instr_labels, "Instruction-based masking", tokenizer)
    h += "</div>"
    h += '<div style="flex:1; min-width:400px;">'
    h += visualize_labels(token_ids, span_labels, "Span-based masking (end_of_turn_template)", tokenizer)
    h += "</div></div>"

    # Summary stats
    instr_unmasked = sum(1 for l in instr_labels if l != IGNORE_INDEX)
    span_unmasked = sum(1 for l in span_labels if l != IGNORE_INDEX)
    h += (
        f'<p style="font-family:monospace; font-size:12px; color:#888;">'
        f"Instruction-based unmasked: {instr_unmasked}/{len(instr_labels)} tokens | "
        f"Span-based unmasked: {span_unmasked}/{len(span_labels)} tokens"
        f"</p><hr>"
    )
    display(HTML(f"<h2>{html_mod.escape(name)}</h2>" + h))

## 4. Build Both Collator Modes

Both use `DataCollatorForCompletionOnlyLM` — the difference is which parameters are provided:
- **Instruction-based**: `instruction_template` + `response_template` (existing TRL behavior)
- **Span-based**: `response_template` + `end_of_turn_template` (new tool-aware path)

In [8]:
import warnings

# --- Instruction-based collator (existing TRL behavior) ---
instruction_collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    instruction_template=INSTRUCTION_TEMPLATE,
    tokenizer=tokenizer,
)

# --- Span-based collator (tool-aware, using end_of_turn_template) ---
span_collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    end_of_turn_template=END_OF_TURN,
    tokenizer=tokenizer,
)

print("Both collator modes created successfully.")

Both collator modes created successfully.


## 5. Run All Scenarios

For each scenario, we run both masking modes and display the color-coded label masks side by side.

**What to look for in the tool-call scenarios:**
- Instruction-based masking uses `<|im_start|>user` as the instruction boundary. Since `tool` is not `user`, tool-result tokens between two assistant turns may **leak** into the loss.
- Span-based masking explicitly finds assistant response spans using `response_template` → `end_of_turn_template` and only unmasks those.

In [9]:
for name, token_ids in tokenized.items():
    example = {"input_ids": token_ids}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        instr_batch = instruction_collator.torch_call([example])
        span_batch = span_collator.torch_call([example])

    instr_labels = instr_batch["labels"][0].tolist()
    span_labels = span_batch["labels"][0].tolist()

    show_comparison(name, token_ids, instr_labels, span_labels)

## 6. Detailed Diff: Which Tokens Differ?\n\nShow only the tokens where the two collators disagree on masking.

In [10]:
for name, token_ids in tokenized.items():
    example = {"input_ids": token_ids}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        instr_batch = instruction_collator.torch_call([example])
        span_batch = span_collator.torch_call([example])

    instr_labels = instr_batch["labels"][0].tolist()
    span_labels = span_batch["labels"][0].tolist()

    diffs = []
    for pos, (tid, instr_l, span_l) in enumerate(zip(token_ids, instr_labels, span_labels)):
        if instr_l != span_l:
            decoded = tokenizer.decode([tid])
            instr_status = "MASKED" if instr_l == IGNORE_INDEX else "unmasked"
            span_status = "MASKED" if span_l == IGNORE_INDEX else "unmasked"
            diffs.append(f"  pos {pos:3d} | {decoded!r:20s} (id={tid:5d}) | Instruction: {instr_status:10s} | Span: {span_status}")

    if diffs:
        print(f"\n{'='*80}")
        print(f"  {name}: {len(diffs)} tokens differ")
        print(f"{'='*80}")
        for d in diffs:
            print(d)
    else:
        print(f"\n  {name}: Both masking modes agree on all tokens")


  Simple (no tools): 2 tokens differ
  pos  22 | '<|im_end|>'         (id=151645) | Instruction: unmasked   | Span: MASKED
  pos  23 | '\n'                 (id=  198) | Instruction: unmasked   | Span: MASKED

  Single tool call: 26 tokens differ
  pos  35 | '<|im_end|>'         (id=151645) | Instruction: unmasked   | Span: MASKED
  pos  36 | '\n'                 (id=  198) | Instruction: unmasked   | Span: MASKED
  pos  37 | '<|im_start|>'       (id=151644) | Instruction: unmasked   | Span: MASKED
  pos  38 | 'tool'               (id=14172) | Instruction: unmasked   | Span: MASKED
  pos  39 | '\n'                 (id=  198) | Instruction: unmasked   | Span: MASKED
  pos  40 | '{"'                 (id= 4913) | Instruction: unmasked   | Span: MASKED
  pos  41 | 'temperature'        (id=34558) | Instruction: unmasked   | Span: MASKED
  pos  42 | '":'                 (id=  788) | Instruction: unmasked   | Span: MASKED
  pos  43 | ' '                  (id=  220) | Instruction: unmasked   |

## 7. Bonus: Span-Based with `mask_tool_calls=True`

When `mask_tool_calls=True`, even the assistant's tool-call turns are masked — only the final plain-text answer is trained on.

In [11]:
TOOL_CALL_START = "<tool_call>"

span_collator_mask_tc = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    end_of_turn_template=END_OF_TURN,
    mask_tool_calls=True,
    tool_call_start_template=TOOL_CALL_START,
    tokenizer=tokenizer,
)

# Compare default span-based vs mask_tool_calls=True on the single tool-call scenario
token_ids = tokenized["Single tool call"]
example = {"input_ids": token_ids}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    span_default = span_collator.torch_call([example])
    span_masked = span_collator_mask_tc.torch_call([example])

span_default_labels = span_default["labels"][0].tolist()
span_masked_labels = span_masked["labels"][0].tolist()

h = '<div style="display:flex; gap:20px; flex-wrap:wrap;">'
h += '<div style="flex:1; min-width:400px;">'
h += visualize_labels(token_ids, span_default_labels, "Span-based (default: train on tool calls)", tokenizer)
h += "</div>"
h += '<div style="flex:1; min-width:400px;">'
h += visualize_labels(token_ids, span_masked_labels, "Span-based (mask_tool_calls=True: final answer only)", tokenizer)
h += "</div></div>"

default_unmasked = sum(1 for l in span_default_labels if l != IGNORE_INDEX)
masked_unmasked = sum(1 for l in span_masked_labels if l != IGNORE_INDEX)
h += (
    f'<p style="font-family:monospace; font-size:12px; color:#888;">'
    f"Default unmasked: {default_unmasked}/{len(span_default_labels)} tokens | "
    f"mask_tool_calls unmasked: {masked_unmasked}/{len(span_masked_labels)} tokens"
    f"</p>"
)
display(HTML(h))